In [4]:
import sys
sys.path.append('../')
from utils_jax import *
import optax
dq.set_device('cpu')
import warnings
warnings.filterwarnings("ignore")

In [21]:
n_lvls_fluxonium = 25
n_lvls_transmon = 4


Ej_f = 3
Ec_f = 3/4
El_f = 3/20.5

qsf = qs.Fluxonium.create(
    n_lvls_fluxonium,
    {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
    )


def objective(params):    


    Ej_t = params[0]
    Ec_t = params[1]
    g_tf = 0.1
    
    qst = MyTransmon.create(
        N = n_lvls_transmon,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=8
        )

    devices = [qsf, qst]
    f_indx = 0
    t_indx = 1
    Ns = [device.N for device in devices]
    fn = qs.promote(qsf.ops["n"], f_indx, Ns)
    tn = qs.promote(qst.ops['n'], t_indx, Ns)

    system = qs.System.create(devices, couplings=[
        g_tf *  fn @ tn
        ])
    system.params["g_tf"] = g_tf
    system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
    driven_op = transform_op_into_dressed_basis_jax(fn, system_evecs_sorted.T)


    matrix_element_0001 = driven_op[
        find_closest_dressed_index( 0*n_lvls_transmon + 0, product_indices_sorted_by_eval),
        find_closest_dressed_index( 0*n_lvls_transmon + 1, product_indices_sorted_by_eval)
    ]
    matrix_element_1011 = driven_op[
        find_closest_dressed_index( 1*n_lvls_transmon + 0, product_indices_sorted_by_eval),
        find_closest_dressed_index( 1*n_lvls_transmon + 1, product_indices_sorted_by_eval)
    ]
    matrix_element_2021 = driven_op[
        find_closest_dressed_index( 2*n_lvls_transmon + 0, product_indices_sorted_by_eval),
        find_closest_dressed_index( 2*n_lvls_transmon + 1, product_indices_sorted_by_eval)
    ]
    diff01 =  jnp.abs(matrix_element_0001-matrix_element_1011)
    diff12 =  jnp.abs(matrix_element_1011- matrix_element_2021)
    return - jnp.log(diff01 + 0.1) - jnp.log(1 - diff12)*50

In [22]:
params = jnp.array([9.0491724 , 0.2483548 ])

func = jax.value_and_grad(objective)
optimizer = optax.nadam(learning_rate=jnp.array([1e-3,
                                                 1e-3])) 
opt_state = optimizer.init( params )

num_steps = 1000
for step in range(num_steps):
    print(f"iter: {step}, params: {params}")
    val, grads = func(params)
    print(f"\t val={val} grads: {grads}\n")
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

iter: 0, params: [9.0491724 0.2483548]
	 val=3.1683918734531464 grads: [ 0.04721987 -0.41558398]

iter: 1, params: [9.04769872 0.24982848]
	 val=4.100010430200596 grads: [0.25705417 4.2396438 ]

iter: 2, params: [9.04642863 0.24866181]
	 val=3.35977729027741 grads: [0.08837754 0.46410091]

iter: 3, params: [9.04563579 0.24817548]
	 val=3.042024107353142 grads: [-0.0172577  -2.29512104]

iter: 4, params: [9.04518715 0.2483367 ]
	 val=3.6599744046842178 grads: [0.10756967 0.36318192]

iter: 5, params: [9.04443582 0.24816456]
	 val=3.9074837682089667 grads: [0.19739508 2.82576515]

iter: 6, params: [9.04350438 0.24758974]
	 val=2.7937200119805037 grads: [-0.06721497 -3.18243804]

iter: 7, params: [9.04311419 0.24780237]
	 val=4.068617925180228 grads: [0.1617074  1.27266104]

iter: 8, params: [9.04234709 0.24759407]
	 val=4.581578781150859 grads: [0.24798893 2.97139975]

iter: 9, params: [9.04143828 0.2471306 ]
	 val=3.0433149540570845 grads: [-0.07360673 -4.11008143]

iter: 10, params: [9

KeyboardInterrupt: 